Pull camera-violation tickets (codes 6, 37) for each parking-violations fiscal year dataset, using `fetch_soda` from `data/download.py` so every pull is cached in the local DuckDB and logged in `_provenance`.

Each fiscal year is fetched in its own cell. Datasets are tens of millions of rows before filtering, so keeping the pulls separate means a failed or interrupted run only has to redo the year it stopped on — `fetch_soda` serves any year already in the cache straight from DuckDB instead of hitting the API again.

In [ ]:
import sys
sys.path.append("..")

from data.download import fetch_soda

WHERE = "violation_code IN (7, 36)"

In [ ]:
fetch_soda("2bnn-yakx", "tickets_fy2017", WHERE).shape

In [ ]:
fetch_soda("a5td-mswe", "tickets_fy2018", WHERE).shape

In [ ]:
fetch_soda("faiq-9dfq", "tickets_fy2019", WHERE).shape

In [ ]:
fetch_soda("p7t3-5i9s", "tickets_fy2020", WHERE).shape

In [ ]:
fetch_soda("kvfd-bves", "tickets_fy2021", WHERE).shape

In [ ]:
fetch_soda("7mxj-7a6y", "tickets_fy2022", WHERE).shape

In [ ]:
fetch_soda("869v-vr48", "tickets_fy2023", WHERE).shape

In [ ]:
fetch_soda("8zf9-spf8", "tickets_fy2024", WHERE).shape

In [ ]:
fetch_soda("m5vz-tzqv", "tickets_fy2025", WHERE).shape

In [ ]:
fetch_soda("9mwx-gamw", "tickets_fy2026", WHERE).shape

In [ ]:
fetch_soda("pvqr-7yc4", "tickets_fy2027", WHERE).shape

In [ ]:
# combine each fiscal year in DuckDB (UNION ALL BY NAME handles the extra
# column some years have) instead of pd.concat, so ~20M rows never have to
# sit in pandas all at once
from data.base import get_connection

FISCAL_YEAR_TABLES = [
    "tickets_fy2017",
    "tickets_fy2018",
    "tickets_fy2019",
    "tickets_fy2020",
    "tickets_fy2021",
    "tickets_fy2022",
    "tickets_fy2023",
    "tickets_fy2024",
    "tickets_fy2025",
    "tickets_fy2026",
    "tickets_fy2027",
]

con = get_connection()
try:
    union_sql = " UNION ALL BY NAME ".join(
        f"SELECT * FROM {table}" for table in FISCAL_YEAR_TABLES
    )
    con.execute(f"CREATE OR REPLACE TABLE tickets_all AS {union_sql}")
    row_count = con.execute("SELECT COUNT(*) FROM tickets_all").fetchone()[0]
finally:
    con.close()

row_count